
# Homography Estimation, Panorama Stitching, and AR  



- Build the pipeline: features → matching → DLT + RANSAC → warping → panorama → AR.


## 0. Environment

In [ ]:

import sys, os, json, glob
from typing import Tuple
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt
import time
import pandas as pd

# Set random seed for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

MAX_CANVAS_LONG_SIDE = 6000        # px
MAX_CANVAS_PIXELS    = 40_000_000  # W*H cap (~40 MP)

print('Python:', sys.version)
print('NumPy:', np.__version__)
print('OpenCV:', cv2.__version__)
print('Random seed:', RANDOM_SEED)

plt.rcParams['figure.figsize'] = (8,6)
plt.rcParams['axes.grid'] = True


Python: 3.13.5 (tags/v3.13.5:6cb20a2, Jun 11 2025, 16:15:46) [MSC v.1943 64 bit (AMD64)]
NumPy: 2.2.6
OpenCV: 4.12.0


## 1. Paths and parameters

In [ ]:

DATA_PANO_DIR = 'panorama_dataset'
DATA_AR_DIR   = 'ar_dataset'
OUT_DIR       = 'outputs'
PANO_OUT_DIR  = os.path.join(OUT_DIR, 'panorama_results')
AR_OUT_VIDEO  = os.path.join(OUT_DIR, 'ar_dynamic_result.mp4')

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PANO_OUT_DIR, exist_ok=True)

PARAMS = {
    'detector': 'SIFT',                 # change per run
    'ratio_test': 0.7,                  # Lowe ratio 
    'matcher': 'BF',                   
    'ransac': {'sample_size':4, 'max_iters':3000, 'inlier_thresh':2.0},
}
print(json.dumps(PARAMS, indent=2))


{
  "detector": "SIFT",
  "ratio_test": 0.7,
  "matcher": "BF",
  "ransac": {
    "sample_size": 4,
    "max_iters": 3000,
    "inlier_thresh": 2.0
  }
}


## 2. IO helpers

In [ ]:

def imread_color(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def imwrite_rgb(path, img_rgb):
    bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    cv2.imwrite(path, bgr)


## 3. Features (SIFT/SURF/ORB)

In [ ]:

def create_detector(name='SIFT'):
    n = name.upper()
    if n == 'SIFT':
        if hasattr(cv2, 'SIFT_create'):
            return cv2.SIFT_create(nfeatures=4000)
        raise RuntimeError('SIFT not available')
    if n == 'SURF':
        if hasattr(cv2, 'xfeatures2d'):
            return cv2.xfeatures2d.SURF_create()
        raise RuntimeError('SURF not available')
    if n == 'ORB':
        return cv2.ORB_create(nfeatures=4000)
    raise ValueError(name)

def method_available(name: str) -> bool:
    n = name.upper()
    if n == 'SIFT':
        return hasattr(cv2, 'SIFT_create')
    if n == 'SURF':
        return hasattr(cv2, 'xfeatures2d') and hasattr(cv2.xfeatures2d, 'SURF_create')
    if n == 'ORB':
        return hasattr(cv2, 'ORB_create')
    return False

def list_enabled_methods(candidates=('SIFT','SURF','ORB')):
    ok = []
    for m in candidates:
        if method_available(m):
            ok.append(m)
        else:
            print(f"[warn] {m} not available in this OpenCV build; skipping.")
    return ok

# Cap the larger side for feature extraction to avoid slowdowns
MAX_FEATURE_IMAGE_DIM = 1400

def _downscale_for_features(img_rgb, max_dim=MAX_FEATURE_IMAGE_DIM):
    h, w = img_rgb.shape[:2]
    m = max(h, w)
    if m <= max_dim:
        return img_rgb, 1.0
    s = max_dim / float(m)
    small = cv2.resize(img_rgb, (int(round(w*s)), int(round(h*s))), interpolation=cv2.INTER_AREA)
    return small, s

def detect_and_describe(img_rgb, method='SIFT'):
    img_small, s = _downscale_for_features(img_rgb)
    gray = cv2.cvtColor(img_small, cv2.COLOR_RGB2GRAY)
    det = create_detector(method)
    kps, desc = det.detectAndCompute(gray, None)
    if kps is None:
        return [], None
    # Map keypoints back to original coordinates
    if s != 1.0:
        for kp in kps:
            kp.pt = (kp.pt[0] / s, kp.pt[1] / s)
            kp.size = kp.size / s
    return kps, desc


## 4. Matching and ratio test

In [ ]:

def create_matcher(method, descriptor_type):
    if method == 'BF':
        if descriptor_type in ('SIFT','SURF'):
            return cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
        else:
            return cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    if method == 'FLANN':
        if descriptor_type in ('SIFT','SURF'):
            index_params = dict(algorithm=1, trees=5)
            search_params = dict(checks=50)
            return cv2.FlannBasedMatcher(index_params, search_params)
        else:
            index_params = dict(algorithm=6, table_number=6, key_size=12, multi_probe_level=1)
            search_params = dict(checks=50)
            return cv2.FlannBasedMatcher(index_params, search_params)
    raise ValueError(method)

def knn_ratio_match(desc1, desc2, k=2, ratio=0.75, matcher='BF', descriptor_type='SIFT', return_raw=False):
    m = create_matcher(matcher, descriptor_type)
    raw = m.knnMatch(desc1, desc2, k=k)
    
    # Extract all raw matches (first match from each pair)
    raw_matches = [pair[0] for pair in raw if len(pair) > 0]
    
    # Apply ratio test
    good = []
    for pair in raw:
        if len(pair) < 2:
            continue
        if pair[0].distance < ratio * pair[1].distance:
            good.append(pair[0])
    
    if return_raw:
        return good, raw_matches
    return good

def matches_to_points(kps1, kps2, matches):
    pts1 = np.float32([kps1[m.queryIdx].pt for m in matches])
    pts2 = np.float32([kps2[m.trainIdx].pt for m in matches])
    return pts1, pts2


## 5. DLT + normalization

In [ ]:
def normalize_points(pts: np.ndarray):
    # Input: pts (N,2). Output: pts_norm (N,2), T (3,3) s.t. x_norm ≈ T @ x
    pts = np.asarray(pts, dtype=np.float64)
    if pts.ndim != 2 or pts.shape[1] != 2:
        raise ValueError("pts must be (N,2)")
    N = pts.shape[0]
    if N < 2:
        return pts.copy(), np.eye(3, dtype=np.float64)

    c = pts.mean(axis=0)
    shifted = pts - c
    mean_dist = np.linalg.norm(shifted, axis=1).mean()
    s = 1.0 if mean_dist < 1e-12 else (np.sqrt(2.0) / mean_dist)

    T = np.array([[s, 0, -s*c[0]],
                  [0, s, -s*c[1]],
                  [0, 0, 1]], dtype=np.float64)

    pts_h = np.hstack([pts, np.ones((N, 1), dtype=np.float64)])
    ptsn_h = (T @ pts_h.T).T
    w = np.where(np.abs(ptsn_h[:, 2]) < 1e-12, 1e-12, ptsn_h[:, 2])
    pts_norm = ptsn_h[:, :2] / w[:, None]
    return pts_norm, T


def dlt_homography(pts1: np.ndarray, pts2: np.ndarray):
    # Input: normalized 2D pairs (N,2). Output: H (3,3) with x2 ~ H x1
    pts1 = np.asarray(pts1, dtype=np.float64)
    pts2 = np.asarray(pts2, dtype=np.float64)
    if pts1.shape != pts2.shape or pts1.ndim != 2 or pts1.shape[1] != 2:
        raise ValueError("pts1 and pts2 must be (N,2) with equal shapes")
    N = pts1.shape[0]
    if N < 4:
        raise ValueError("At least 4 correspondences are required")

    A = np.zeros((2*N, 9), dtype=np.float64)
    for i, ((x, y), (u, v)) in enumerate(zip(pts1, pts2)):
        A[2*i]   = [-x, -y, -1,  0,  0,  0,  u*x, u*y, u]
        A[2*i+1] = [ 0,  0,  0, -x, -y, -1,  v*x, v*y, v]

    _, _, Vt = np.linalg.svd(A, full_matrices=True)
    H = Vt[-1].reshape(3, 3)
    if np.abs(H[2, 2]) > 1e-12:
        H = H / H[2, 2]
    else:
        H = H / (np.linalg.norm(H) + 1e-12)
    return H


def estimate_homography(pts1: np.ndarray, pts2: np.ndarray):
    # normalize -> DLT -> denormalize
    pts1 = np.asarray(pts1, dtype=np.float64)
    pts2 = np.asarray(pts2, dtype=np.float64)
    if pts1.shape != pts2.shape or pts1.ndim != 2 or pts1.shape[1] != 2:
        raise ValueError("pts1 and pts2 must be (N,2) with equal shapes")

    p1n, T1 = normalize_points(pts1)
    p2n, T2 = normalize_points(pts2)
    Hn = dlt_homography(p1n, p2n)
    H = np.linalg.inv(T2) @ Hn @ T1

    if np.abs(H[2, 2]) > 1e-12:
        H = H / H[2, 2]
    else:
        H = H / (np.linalg.norm(H) + 1e-12)
    return H


## 6. RANSAC

In [ ]:
def ransac_homography(pts1: np.ndarray, pts2: np.ndarray,
                      sample_size=4, max_iters=2000, inlier_thresh=3.0, rng=None):
    """
    Wstimate homography with RANSAC.
    Returns:
      H_best (3,3) and inliers_mask of type bool.
    """
    pts1 = np.asarray(pts1, dtype=np.float64)
    pts2 = np.asarray(pts2, dtype=np.float64)
    if pts1.shape != pts2.shape or pts1.ndim != 2 or pts1.shape[1] != 2:
        raise ValueError("pts1 and pts2 must be (N,2) with equal shapes")
    N = pts1.shape[0]
    if N < sample_size:
        return None, np.zeros(N, dtype=bool)

    if rng is None:
        rng = np.random.default_rng()

    def to_h(pts):
        return np.hstack([pts, np.ones((pts.shape[0], 1), dtype=np.float64)])

    X1h = to_h(pts1) 
    X2h = to_h(pts2)

    best_H = None
    best_inliers = np.zeros(N, dtype=bool)
    best_inlier_count = 0

    for _ in range(max_iters):
        idx = rng.choice(N, size=sample_size, replace=False)
        try:
            H = estimate_homography(pts1[idx], pts2[idx])
        except Exception:
            continue

        # forward projection: pts1 -> pts2
        proj2 = (H @ X1h.T).T
        w = np.where(np.abs(proj2[:, 2]) < 1e-12, 1e-12, proj2[:, 2])
        proj2_xy = proj2[:, :2] / w[:, None]
        e1 = np.sum((proj2_xy - pts2)**2, axis=1)

        # backward projection: pts2 -> pts1
        try:
            Hinv = np.linalg.inv(H)
            proj1 = (Hinv @ X2h.T).T
            w1 = np.where(np.abs(proj1[:, 2]) < 1e-12, 1e-12, proj1[:, 2])
            proj1_xy = proj1[:, :2] / w1[:, None]
            e2 = np.sum((proj1_xy - pts1)**2, axis=1)
        except np.linalg.LinAlgError:
            e2 = np.full(N, np.inf, dtype=np.float64)

        # symmetric transfer error
        err = e1 + e2
        inliers = err <= (inlier_thresh ** 2) * 2.0 

        cnt = int(inliers.sum())
        if cnt > best_inlier_count:
            best_inlier_count = cnt
            best_inliers = inliers
            best_H = H

    # Refit using all inliers for the best model
    if best_inlier_count >= sample_size:
        try:
            best_H = estimate_homography(pts1[best_inliers], pts2[best_inliers])
        except Exception:
            pass
    else:
        best_H = None  # no reliable model

    return best_H, best_inliers


## 7. Warping and simple blending

In [232]:

def warp_image(img_rgb, H, out_shape: Tuple[int,int]):
    return cv2.warpPerspective(img_rgb, H, out_shape, flags=cv2.INTER_LINEAR)

def simple_linear_blend(base, overlay, mask):
    base = base.copy()
    alpha = cv2.GaussianBlur((mask.astype(np.float32)), (31,31), 0)
    maxv = alpha.max() if alpha.max() > 0 else 1.0
    alpha = alpha / maxv
    if base.ndim == 3:
        alpha = np.stack([alpha]*3, axis=-1)
    out = (1 - alpha)*base + alpha*overlay
    return np.clip(out, 0, 255).astype(np.uint8)


## 8. Panorama stitching pipeline

In [ ]:
def compute_pairwise_homography(img1, img2, method='SIFT'):
    """Compute H that maps img2 -> img1"""
    k1, d1 = detect_and_describe(img1, method=method)
    k2, d2 = detect_and_describe(img2, method=method)
    
    if len(k1) < 4 or len(k2) < 4:
        return None, 0, None, None, None, None
    
    matches = knn_ratio_match(d1, d2, k=2, ratio=PARAMS['ratio_test'],
                              matcher=PARAMS['matcher'], descriptor_type=method)
    if len(matches) < 4:
        return None, 0, None, None, None, None
    
    pts1, pts2 = matches_to_points(k1, k2, matches)
    H, inliers = ransac_homography(pts2, pts1,
                                   sample_size=PARAMS['ransac']['sample_size'],
                                   max_iters=PARAMS['ransac']['max_iters'],
                                   inlier_thresh=PARAMS['ransac']['inlier_thresh'])
    
    inlier_count = inliers.sum() if inliers is not None else 0
    return H, inlier_count, k1, k2, matches, inliers


def visualize_keypoints(img, kps, title='Keypoints', max_display=1000):
    """Draw keypoints on image with size and orientation"""
    # Limit display for performance
    if len(kps) > max_display:
        kps_display = random.sample(kps, max_display)
    else:
        kps_display = kps
    out = cv2.drawKeypoints(img, kps_display, None, 
                           color=(0, 255, 0),
                           flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    return out


def draw_line_matches(img1, kps1, img2, kps2, matches, colors=None):
    """Draw matches with alternating colors"""
    if colors is None:
        # Alternating colors
        colors = [
            (255, 0, 0),    
            (0, 255, 0),    
            (0, 100, 255),  
            (255, 255, 0),  
            (0, 255, 255),   
            (255, 0, 255),   
            (255, 128, 0),    
            (128, 0, 255),    
        ]
    
    # Create output image 
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]
    h = max(h1, h2)
    w = w1 + w2
    
    out_img = np.zeros((h, w, 3), dtype=np.uint8)
    out_img[:h1, :w1] = img1
    out_img[:h2, w1:w1+w2] = img2
    
    # Draw each match with alternating colors
    for idx, m in enumerate(matches):
        color = colors[idx % len(colors)]
        
        # Get keypoint positions
        pt1 = tuple(map(int, kps1[m.queryIdx].pt))
        pt2 = tuple(map(int, kps2[m.trainIdx].pt))
        pt2_shifted = (pt2[0] + w1, pt2[1])  # Shift to right image
        cv2.line(out_img, pt1, pt2_shifted, color, 1, cv2.LINE_AA)
        
        # Draw keypoint circles
        cv2.circle(out_img, pt1, 3, color, -1, cv2.LINE_AA)
        cv2.circle(out_img, pt2_shifted, 3, color, -1, cv2.LINE_AA)
    
    return out_img


def visualize_matches_split(img1, kps1, img2, kps2, matches, inliers, 
                            title_inlier='Inlier Matches', title_outlier='Outlier Matches',
                            max_display=100):
    """Visualize inliers and outliers with colorful alternating lines
    """
    if inliers is None:
        return None, None
    
    inlier_matches = [m for i, m in enumerate(matches) if inliers[i]]
    outlier_matches = [m for i, m in enumerate(matches) if not inliers[i]]
    
    # Limit display for visual clarity, randomly sample if too many
    if len(inlier_matches) > max_display:
        inlier_matches = random.sample(inlier_matches, max_display)
    
    if len(outlier_matches) > max_display:
        outlier_matches = random.sample(outlier_matches, max_display)
    
    # Draw inliers 
    img_inliers = None
    if len(inlier_matches) > 0:
        img_inliers = draw_line_matches(img1, kps1, img2, kps2, inlier_matches)
    
    # Draw outliers
    img_outliers = None
    if len(outlier_matches) > 0:
        outlier_colors = [(255, 0, 0), (180, 0, 0), (255, 50, 50)]  # Red variations
        img_outliers = draw_line_matches(img1, kps1, img2, kps2, outlier_matches, outlier_colors)
    
    return img_inliers, img_outliers


def visualize_scene_stitching(img_paths, method='SIFT', scene_name='scene'):
    """
    Create comprehensive visualizations for panorama stitching pipeline.
    Saves keypoint visualizations and match visualizations for each pair.
    """
    images = [imread_color(p) for p in img_paths]
    n = len(images)
    
    # Create visualization output directory
    vis_dir = os.path.join(OUT_DIR, 'visualizations', scene_name, method.lower())
    os.makedirs(vis_dir, exist_ok=True)
    
    print(f"\n{'='*60}")
    print(f"Visualizing {scene_name} with {method}")
    print(f"{'='*60}")
    
    # Visualize keypoints for each image
    print("\n1. Detecting and visualizing keypoints...")
    for i, img in enumerate(images):
        kps, desc = detect_and_describe(img, method=method)
        print(f"  Image {i+1}: {len(kps)} keypoints detected")
        
        kp_img = visualize_keypoints(img, kps, title=f'Image {i+1} Keypoints')
        kp_path = os.path.join(vis_dir, f'keypoints_img{i+1}.png')
        imwrite_rgb(kp_path, kp_img)
    
    # Visualize before/after ratio test (first pair only)
    print("\n2. Visualizing effect of ratio test (first pair)...")
    if n >= 2:
        k1, d1 = detect_and_describe(images[0], method=method)
        k2, d2 = detect_and_describe(images[1], method=method)
        
        # Get both raw and filtered matches
        filtered_matches, raw_matches = knn_ratio_match(
            d1, d2, k=2, ratio=PARAMS['ratio_test'],
            matcher=PARAMS['matcher'], descriptor_type=method, return_raw=True
        )
        
        print(f"  Raw matches (before ratio test): {len(raw_matches)}")
        print(f"  Filtered matches (after ratio test): {len(filtered_matches)}")
        print(f"  Filtered out: {len(raw_matches) - len(filtered_matches)} ({(len(raw_matches)-len(filtered_matches))/len(raw_matches)*100:.1f}%)")
        print(f"  Note: Showing ALL matches to visualize the filtering effect")
        
        # Visualize before ratio test
        if len(raw_matches) > 0:
            img_before = draw_line_matches(images[0], k1, images[1], k2, raw_matches)
            before_path = os.path.join(vis_dir, 'matches_before_ratio_test.png')
            imwrite_rgb(before_path, img_before)
        
        # Visualize after ratio test
        if len(filtered_matches) > 0:
            img_after = draw_line_matches(images[0], k1, images[1], k2, filtered_matches)
            after_path = os.path.join(vis_dir, 'matches_after_ratio_test.png')
            imwrite_rgb(after_path, img_after)
    
    # Visualize pairwise matches with inliers/outliers
    print("\n3. Computing matches and visualizing inliers/outliers...")
    for i in range(n - 1):
        print(f"\n  Pair: Image {i+1} ↔ Image {i+2}")
        
        H, inlier_cnt, k1, k2, matches, inliers = compute_pairwise_homography(
            images[i], images[i+1], method=method
        )
        
        if H is None or matches is None:
            print(f"    Warning: Failed to compute matches")
            continue
        
        num_outliers = len(matches) - inlier_cnt
        print(f"    Total matches: {len(matches)}")
        print(f"    Inliers: {inlier_cnt} ({inlier_cnt/len(matches)*100:.1f}%)")
        print(f"    Outliers: {num_outliers} ({num_outliers/len(matches)*100:.1f}%)")
        
        max_display = 100
        if inlier_cnt > max_display or num_outliers > max_display:
            print(f"    Note: Displaying random sample of {max_display} matches for clarity")
        
        img_inliers, img_outliers = visualize_matches_split(
            images[i], k1, images[i+1], k2, matches, inliers, max_display=max_display
        )
        
        if img_inliers is not None:
            inlier_path = os.path.join(vis_dir, f'inliers_pair{i+1}_{i+2}.png')
            imwrite_rgb(inlier_path, img_inliers)
        
        if img_outliers is not None:
            outlier_path = os.path.join(vis_dir, f'outliers_pair{i+1}_{i+2}.png')
            imwrite_rgb(outlier_path, img_outliers)
    
    print(f"\nVisualizations saved to: {vis_dir}\n")


def stitch_scene_with_method(img_paths, method: str):
    """
    Stitch images using reference-based approach with improved blending.
    Uses first image as reference to maintain natural orientation.
    """
    images = [imread_color(p) for p in img_paths]
    n = len(images)
    if n < 2:
        return images[0], 0, 0
    
    # Use first image as reference 
    ref_idx = 0
    
    # Compute homographies for all images relative to reference
    H_to_ref = [None] * n
    H_to_ref[ref_idx] = np.eye(3, dtype=np.float64)
    total_inliers = 0
    
    # Accumulate homographies from left to right
    for i in range(1, n):
        H_i_to_im1, inliers_count, _, _, _, _ = compute_pairwise_homography(images[i-1], images[i], method=method)
        if H_i_to_im1 is None:
            print(f"  Warning: Failed to compute H from image {i+1} to {i}")
            H_to_ref[i] = H_to_ref[i-1].copy()  # fallback: use previous
        else:
            H_to_ref[i] = H_to_ref[i-1] @ H_i_to_im1
            total_inliers += inliers_count
    
    # Compute canvas bounds
    all_corners = []
    for img, H in zip(images, H_to_ref):
        h, w = img.shape[:2]
        corners = np.array([[0,0],[w,0],[w,h],[0,h]], dtype=np.float32)
        corners_t = cv2.perspectiveTransform(corners[None, :, :], H.astype(np.float32))[0]
        all_corners.append(corners_t)
    
    all_corners = np.vstack(all_corners)
    xmin, ymin = np.floor(all_corners.min(axis=0)).astype(float)
    xmax, ymax = np.ceil(all_corners.max(axis=0)).astype(float)
    
    # Constrain canvas to reasonable bounds to handle accumulation errors
    # Instead of rejecting homographies, we limit the canvas size
    ref_img_h, ref_img_w = images[ref_idx].shape[:2]
    max_reasonable_size = max(ref_img_h, ref_img_w) * n * 2
    
    W_raw = xmax - xmin
    H_raw = ymax - ymin
    
    if W_raw > max_reasonable_size or H_raw > max_reasonable_size:
        # Clamp bounds to center around the reference image region
        ref_corners = np.array([[0,0],[ref_img_w,0],[ref_img_w,ref_img_h],[0,ref_img_h]], dtype=np.float32)
        ref_center = ref_corners.mean(axis=0)
        
        # Limit expansion from center
        half_size = max_reasonable_size / 2
        xmin = max(xmin, ref_center[0] - half_size)
        xmax = min(xmax, ref_center[0] + half_size)
        ymin = max(ymin, ref_center[1] - half_size)
        ymax = min(ymax, ref_center[1] + half_size)
        
        print(f"  Note: Canvas bounds constrained to prevent excessive size ({W_raw:.0f}x{H_raw:.0f} -> {xmax-xmin:.0f}x{ymax-ymin:.0f})")
    
    # Translation to positive coordinates
    tx, ty = -min(0.0, xmin), -min(0.0, ymin)
    T = np.array([[1,0,tx],[0,1,ty],[0,0,1]], dtype=np.float64)
    
    # Canvas size
    W = xmax - min(0.0, xmin)
    Hh = ymax - min(0.0, ymin)
    out_w = int(np.ceil(W))
    out_h = int(np.ceil(Hh))
    
    s1 = MAX_CANVAS_LONG_SIDE / max(out_w, out_h) if max(out_w, out_h) > 0 else 1.0
    s2 = np.sqrt(MAX_CANVAS_PIXELS / (out_w * out_h)) if (out_w * out_h) > 0 else 1.0
    s = min(1.0, s1, s2)
    s = float(max(1e-4, s))
    
    S = np.array([[s,0,0],[0,s,0],[0,0,1]], dtype=np.float64)
    out_w_s = max(1, int(np.ceil(out_w * s)))
    out_h_s = max(1, int(np.ceil(Hh * s)))
    
    # Warp all images to canvas
    canvas = np.zeros((out_h_s, out_w_s, 3), dtype=np.float32)
    weight_sum = np.zeros((out_h_s, out_w_s, 1), dtype=np.float32)
    
    for i, (img, H) in enumerate(zip(images, H_to_ref)):
        H_total = (S @ T @ H).astype(np.float32)
        warped = cv2.warpPerspective(img, H_total, (out_w_s, out_h_s))
        
        mask = (warped.sum(axis=2) > 0).astype(np.uint8)
        if mask.sum() == 0:
            continue 
        
        # Distance transform for blending
        dist = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
        
        max_dist = dist.max()
        if max_dist > 1e-6:
            weight = dist / max_dist
        else:
            weight = dist
        
        weight = cv2.GaussianBlur(weight, (71, 71), 0)
        weight = np.maximum(weight, 0.005) 
        weight = weight[:, :, None]
        
        canvas += warped.astype(np.float32) * weight
        weight_sum += weight
    
    # Normalize by total weight
    weight_sum = np.maximum(weight_sum, 1e-6)
    pano = np.clip(canvas / weight_sum, 0, 255).astype(np.uint8)
    
    return pano, total_inliers, (n - 1)


In [ ]:
def compute_homography_between(img1, img2, method='SIFT'):
    """Compute homography from img2 to img1 (without stitching)"""
    k1, d1 = detect_and_describe(img1, method=method)
    k2, d2 = detect_and_describe(img2, method=method)
    
    if len(k1) < 10 or len(k2) < 10:
        print(f"  Warning: Few keypoints detected ({len(k1)}, {len(k2)})")
        return None, 0
    
    matches = knn_ratio_match(d1, d2, k=2, ratio=PARAMS['ratio_test'], 
                              matcher=PARAMS['matcher'], descriptor_type=method)
    
    if len(matches) < 20:
        print(f"  Warning: Only {len(matches)} matches found")
        return None, 0
    
    pts1, pts2 = matches_to_points(k1, k2, matches)
    
    H, inliers = ransac_homography(pts2, pts1,
                                   sample_size=4,
                                   max_iters=5000,  
                                   inlier_thresh=5.0)  
    
    inlier_count = inliers.sum() if inliers is not None else 0
    inlier_ratio = inlier_count / len(matches) if len(matches) > 0 else 0
    
    print(f"  Matches: {len(matches)}, Inliers: {inlier_count} ({inlier_ratio:.1%})")
    
    # Validate homography quality
    if H is None or inlier_count < 15:
        print(f"  Warning: Poor homography quality")
        return None, 0
    
    return H, inlier_count


## 9a. Visualization Tools


In [ ]:
def list_scenes(root):
    if not os.path.isdir(root):
        print('Dir not found:', root)
        return []
    return [d for d in sorted(os.listdir(root)) if os.path.isdir(os.path.join(root, d))]

def visualize_one_scene(scene_name='v_bird', method='SIFT'):
    """Generate detailed visualizations for a single scene"""
    scene_dir = os.path.join(DATA_PANO_DIR, scene_name)
    img_paths = sorted(glob.glob(os.path.join(scene_dir, '*.png')) + 
                      glob.glob(os.path.join(scene_dir, '*.jpg')))
    
    if len(img_paths) < 2:
        print(f'Scene {scene_name} has insufficient images')
        return
    
    visualize_scene_stitching(img_paths, method=method, scene_name=scene_name)


def visualize_all_scenes(scenes=None, methods=('SIFT', 'ORB')):
    """Generate visualizations for all scenes"""
    if scenes is None:
        scenes = list_scenes(DATA_PANO_DIR)
    
    enabled = list_enabled_methods(methods)
    
    for scene in scenes:
        scene_dir = os.path.join(DATA_PANO_DIR, scene)
        img_paths = sorted(glob.glob(os.path.join(scene_dir, '*.png')) + 
                          glob.glob(os.path.join(scene_dir, '*.jpg')))
        
        if len(img_paths) < 2:
            print(f'Skipping {scene}: not enough images')
            continue
        
        for method in enabled:
            visualize_scene_stitching(img_paths, method=method, scene_name=scene)


# Visualize inliers, keypoints and outliers in all scenes
visualize_all_scenes(methods=('SIFT', 'ORB'))




Visualizing v_bird with SIFT

1. Detecting and visualizing keypoints...
  Image 1: 3361 keypoints detected
  Image 2: 3200 keypoints detected
  Image 3: 2703 keypoints detected
  Image 4: 3708 keypoints detected
  Image 5: 4001 keypoints detected
  Image 6: 2344 keypoints detected

2. Visualizing effect of ratio test (first pair)...
  Raw matches (before ratio test): 3361
  Filtered matches (after ratio test): 1040
  Filtered out: 2321 (69.1%)
  Note: Showing ALL matches to visualize the filtering effect

3. Computing matches and visualizing inliers/outliers...

  Pair: Image 1 ↔ Image 2
    Total matches: 1040
    Inliers: 901 (86.6%)
    Outliers: 139 (13.4%)
    Note: Displaying random sample of 100 matches for clarity

  Pair: Image 2 ↔ Image 3
    Total matches: 763
    Inliers: 625 (81.9%)
    Outliers: 138 (18.1%)
    Note: Displaying random sample of 100 matches for clarity

  Pair: Image 3 ↔ Image 4
    Total matches: 334
    Inliers: 226 (67.7%)
    Outliers: 108 (32.3%)
   

In [ ]:
def visualize_warping_process(img_paths, method='SIFT', scene_name='scene'):
    """
    Generate intermediate visualizations for the warping and panorama process:
    1. Warped image overlay with transparency
    2. Overlap mask showing coverage
    3. Final panorama
    """
    images = [imread_color(p) for p in img_paths]
    n = len(images)
    if n < 2:
        return
    
    # Create output directory
    warp_vis_dir = os.path.join(OUT_DIR, 'warping_visualization', scene_name, method.lower())
    os.makedirs(warp_vis_dir, exist_ok=True)
    
    print(f"\nGenerating warping visualizations for {scene_name} ({method})...")
    
    ref_idx = 0
    H_to_ref = [None] * n
    H_to_ref[ref_idx] = np.eye(3, dtype=np.float64)
    
    # Compute homographies
    for i in range(1, n):
        H_i_to_im1, inliers_count, _, _, _, _ = compute_pairwise_homography(images[i-1], images[i], method=method)
        if H_i_to_im1 is None:
            H_to_ref[i] = H_to_ref[i-1].copy()
        else:
            H_to_ref[i] = H_to_ref[i-1] @ H_i_to_im1
    
    # Compute canvas bounds 
    all_corners = []
    for img, H in zip(images, H_to_ref):
        h, w = img.shape[:2]
        corners = np.array([[0,0],[w,0],[w,h],[0,h]], dtype=np.float32)
        corners_t = cv2.perspectiveTransform(corners[None, :, :], H.astype(np.float32))[0]
        all_corners.append(corners_t)
    
    all_corners = np.vstack(all_corners)
    xmin, ymin = np.floor(all_corners.min(axis=0)).astype(float)
    xmax, ymax = np.ceil(all_corners.max(axis=0)).astype(float)
    
    # Constrain canvas 
    ref_img_h, ref_img_w = images[ref_idx].shape[:2]
    max_reasonable_size = max(ref_img_h, ref_img_w) * n * 2
    
    W_raw = xmax - xmin
    H_raw = ymax - ymin
    
    if W_raw > max_reasonable_size or H_raw > max_reasonable_size:
        ref_corners = np.array([[0,0],[ref_img_w,0],[ref_img_w,ref_img_h],[0,ref_img_h]], dtype=np.float32)
        ref_center = ref_corners.mean(axis=0)
        half_size = max_reasonable_size / 2
        xmin = max(xmin, ref_center[0] - half_size)
        xmax = min(xmax, ref_center[0] + half_size)
        ymin = max(ymin, ref_center[1] - half_size)
        ymax = min(ymax, ref_center[1] + half_size)
    
    tx, ty = -min(0.0, xmin), -min(0.0, ymin)
    T = np.array([[1,0,tx],[0,1,ty],[0,0,1]], dtype=np.float64)
    
    W = xmax - min(0.0, xmin)
    Hh = ymax - min(0.0, ymin)
    out_w = int(np.ceil(W))
    out_h = int(np.ceil(Hh))
    
    s1 = MAX_CANVAS_LONG_SIDE / max(out_w, out_h) if max(out_w, out_h) > 0 else 1.0
    s2 = np.sqrt(MAX_CANVAS_PIXELS / (out_w * out_h)) if (out_w * out_h) > 0 else 1.0
    s = min(1.0, s1, s2)
    s = float(max(1e-4, s))
    
    S = np.array([[s,0,0],[0,s,0],[0,0,1]], dtype=np.float64)
    out_w_s = max(1, int(np.ceil(out_w * s)))
    out_h_s = max(1, int(np.ceil(Hh * s)))
    
    # 1. Create warped image overlay with transparency
    print("  1. Creating warped image overlay...")
    overlay = np.zeros((out_h_s, out_w_s, 3), dtype=np.float32)
    overlap_count_for_overlay = np.zeros((out_h_s, out_w_s), dtype=np.uint8)
    
    # Define distinct colors for each image
    colors = [
        [1.0, 0.2, 0.2],
        [0.2, 1.0, 0.2],  
        [0.2, 0.2, 1.0],  
        [1.0, 1.0, 0.2], 
        [1.0, 0.2, 1.0], 
        [0.2, 1.0, 1.0],  
    ]
    
    for i, (img, H) in enumerate(zip(images, H_to_ref)):
        H_total = (S @ T @ H).astype(np.float32)
        warped = cv2.warpPerspective(img, H_total, (out_w_s, out_h_s))
        mask = (warped.sum(axis=2) > 0).astype(np.uint8)
        
        if mask.sum() > 0:
            color = colors[i % len(colors)]
            tinted = warped.astype(np.float32) * 0.5 + np.array(color) * 128
            
            # Add with transparency
            alpha = 0.4 
            overlay += tinted * alpha * mask[:, :, None]
            overlap_count_for_overlay += mask
    
    # Normalize and save overlay
    overlay_vis = np.clip(overlay / max(1, overlap_count_for_overlay.max()), 0, 255).astype(np.uint8)
    overlay_path = os.path.join(warp_vis_dir, 'warped_overlay.png')
    imwrite_rgb(overlay_path, overlay_vis)
    
    # Create overlap mask visualization
    print("  2. Creating overlap mask...")
    # Create a heatmap showing how many images overlap at each pixel
    overlap_count = np.zeros((out_h_s, out_w_s), dtype=np.uint8)
    overlap_heatmap = np.zeros((out_h_s, out_w_s, 3), dtype=np.uint8)
    
    # Count overlaps
    for i, (img, H) in enumerate(zip(images, H_to_ref)):
        H_total = (S @ T @ H).astype(np.float32)
        warped = cv2.warpPerspective(img, H_total, (out_w_s, out_h_s))
        mask = (warped.sum(axis=2) > 0).astype(np.uint8)
        overlap_count += mask
    
    overlap_heatmap[overlap_count == 1] = [0, 0, 255]      # Blue - single coverage
    overlap_heatmap[overlap_count == 2] = [0, 255, 0]      # Green - double coverage
    overlap_heatmap[overlap_count == 3] = [255, 255, 0]    # Yellow - triple coverage
    overlap_heatmap[overlap_count >= 4] = [255, 0, 0]      # Red - 4+ coverage
    
    overlap_path = os.path.join(warp_vis_dir, 'overlap_mask.png')
    imwrite_rgb(overlap_path, overlap_heatmap)
    
    # Generate final panorama (for reference)
    print("  3. Creating final panorama...")
    pano, _, _ = stitch_scene_with_method(img_paths, method)
    pano_path = os.path.join(warp_vis_dir, 'final_panorama.png')
    imwrite_rgb(pano_path, pano)
    
    print(f"  Saved to: {warp_vis_dir}")
    print(f"    - warped_overlay.png (transparent overlay)")
    print(f"    - overlap_mask.png (coverage heatmap)")
    print(f"    - final_panorama.png (blended result)")
    
    return overlay_vis, overlap_heatmap, pano


# Generate warping visualizations for all scenes
def visualize_warping_all_scenes(scenes=None, methods=('SIFT', 'ORB')):
    """Generate warping process visualizations for all scenes"""
    if scenes is None:
        scenes = list_scenes(DATA_PANO_DIR)
    
    enabled = list_enabled_methods(methods)
    
    for scene in scenes:
        scene_dir = os.path.join(DATA_PANO_DIR, scene)
        img_paths = sorted(glob.glob(os.path.join(scene_dir, '*.png')) + 
                          glob.glob(os.path.join(scene_dir, '*.jpg')))
        
        if len(img_paths) < 2:
            print(f'Skipping {scene}: not enough images')
            continue
        
        for method in enabled:
            visualize_warping_process(img_paths, method=method, scene_name=scene)

visualize_warping_all_scenes(methods=('SIFT', 'ORB'))



Generating warping visualizations for v_bird (SIFT)...
  1. Creating warped image overlay...
  2. Creating overlap mask...
  3. Creating final panorama...
  Saved to: outputs\warping_visualization\v_bird\sift
    - warped_overlay.png (transparent overlay)
    - overlap_mask.png (coverage heatmap)
    - final_panorama.png (blended result)

Generating warping visualizations for v_bird (ORB)...
  1. Creating warped image overlay...
  2. Creating overlap mask...
  3. Creating final panorama...
  Note: Canvas bounds constrained to prevent excessive size (18125x7975 -> 7882x7975)
  Saved to: outputs\warping_visualization\v_bird\orb
    - warped_overlay.png (transparent overlay)
    - overlap_mask.png (coverage heatmap)
    - final_panorama.png (blended result)

Generating warping visualizations for v_boat (SIFT)...
  1. Creating warped image overlay...
  2. Creating overlap mask...
  3. Creating final panorama...
  Saved to: outputs\warping_visualization\v_boat\sift
    - warped_overlay.png

## 9b. Batch over scenes

In [ ]:

def run_all_scenes_all_methods(methods=('SIFT','SURF','ORB')):
    enabled = list_enabled_methods(methods)
    if not enabled:
        raise RuntimeError("No feature methods available (SIFT/SURF/ORB). Install opencv-contrib-python for SIFT/SURF.")

    os.makedirs(PANO_OUT_DIR, exist_ok=True)
    scenes = list_scenes(DATA_PANO_DIR)
    print("Scenes:", scenes)

    logs = []
    for scene in scenes:
        scene_dir = os.path.join(DATA_PANO_DIR, scene)
        img_paths = sorted(glob.glob(os.path.join(scene_dir, '*.png')) +
                           glob.glob(os.path.join(scene_dir, '*.jpg')))
        if len(img_paths) < 2:
            print(f"[skip] {scene}: not enough images")
            continue

        for m in enabled:
            print(f"[{scene}] method={m} ...")
            t0 = time.time()
            try:
                pano, total_inliers, pairs = stitch_scene_with_method(img_paths, m)
                dt = time.time() - t0

                out_name = f"{scene}_{m.lower()}_panorama.png"
                out_path = os.path.join(PANO_OUT_DIR, out_name)
                imwrite_rgb(out_path, pano)
                print(f"  saved -> {out_path} ({dt:.2f}s, inliers={total_inliers}, pairs={pairs})")

                logs.append({
                    'scene': scene,
                    'method': m,
                    'runtime_s': round(dt, 3),
                    'total_inliers': int(total_inliers),
                    'pairs': int(pairs),
                    'inliers_per_pair': round(total_inliers / max(1, pairs), 2),
                    'output': out_name
                })

            except Exception as e:
                dt = time.time() - t0
                print(f"  [error] {scene} {m}: {e} ({dt:.2f}s)")
                logs.append({
                    'scene': scene,
                    'method': m,
                    'runtime_s': round(dt, 3),
                    'total_inliers': None,
                    'pairs': len(img_paths)-1,
                    'inliers_per_pair': None,
                    'output': None,
                    'error': str(e)
                })

    df = pd.DataFrame(logs)
    csv_path = os.path.join(OUT_DIR, "panorama_benchmark_methods.csv")
    df.to_csv(csv_path, index=False)
    print(f"\nBenchmark CSV saved -> {csv_path}")
    try:
        from IPython.display import display
        display(df)
    except Exception:
        pass

run_all_scenes_all_methods(('SIFT','ORB'))


Scenes: ['v_bird', 'v_boat', 'v_circus', 'v_graffiti', 'v_soldiers', 'v_weapons']
[v_bird] method=SIFT ...
  saved -> outputs\panorama_results\v_bird_sift_panorama.png (13.63s, inliers=2668, pairs=5)
[v_bird] method=ORB ...
  saved -> outputs\panorama_results\v_bird_orb_panorama.png (13.97s, inliers=2072, pairs=5)
[v_boat] method=SIFT ...
  saved -> outputs\panorama_results\v_boat_sift_panorama.png (8.93s, inliers=5224, pairs=5)
[v_boat] method=ORB ...
  saved -> outputs\panorama_results\v_boat_orb_panorama.png (7.32s, inliers=5419, pairs=5)
[v_circus] method=SIFT ...
  saved -> outputs\panorama_results\v_circus_sift_panorama.png (7.25s, inliers=4145, pairs=5)
[v_circus] method=ORB ...
  saved -> outputs\panorama_results\v_circus_orb_panorama.png (7.06s, inliers=2386, pairs=5)
[v_graffiti] method=SIFT ...
  Note: Canvas bounds constrained to prevent excessive size (14339x1477 -> 5771x1477)
  saved -> outputs\panorama_results\v_graffiti_sift_panorama.png (9.53s, inliers=4605, pairs=5)
[

,scene,method,runtime_s,total_inliers,pairs,inliers_per_pair,output
0,v_bird,SIFT,13.634,2668,5,533.6,v_bird_sift_panorama.png
1,v_bird,ORB,13.967,2072,5,414.4,v_bird_orb_panorama.png
2,v_boat,SIFT,8.932,5224,5,1044.8,v_boat_sift_panorama.png
3,v_boat,ORB,7.322,5419,5,1083.8,v_boat_orb_panorama.png
4,v_circus,SIFT,7.247,4145,5,829.0,v_circus_sift_panorama.png
5,v_circus,ORB,7.065,2386,5,477.2,v_circus_orb_panorama.png
6,v_graffiti,SIFT,9.533,4605,5,921.0,v_graffiti_sift_panorama.png
7,v_graffiti,ORB,8.606,2597,5,519.4,v_graffiti_orb_panorama.png
8,v_soldiers,SIFT,5.578,666,5,133.2,v_soldiers_sift_panorama.png
9,v_soldiers,ORB,4.264,792,5,158.4,v_soldiers_orb_panorama.png


## Analysis - RANSAC Parameters

In [224]:
# RANSAC - Test different parameters
def ransac_ablation_study(scene_name='v_circus', method='SIFT', img_idx_pair=(0, 1)):
    """
    Test how different RANSAC parameters affect inlier count and ratio.
    Tests on a single image pair.
    """
    scene_dir = os.path.join(DATA_PANO_DIR, scene_name)
    img_paths = sorted(glob.glob(os.path.join(scene_dir, '*.png')) + 
                      glob.glob(os.path.join(scene_dir, '*.jpg')))
    
    if len(img_paths) < 2:
        print(f"Scene {scene_name} has insufficient images")
        return None
    
    img1 = imread_color(img_paths[img_idx_pair[0]])
    img2 = imread_color(img_paths[img_idx_pair[1]])
    
    # Detect features once
    print(f"Testing RANSAC parameters on {scene_name} (images {img_idx_pair[0]+1} and {img_idx_pair[1]+1})")
    k1, d1 = detect_and_describe(img1, method=method)
    k2, d2 = detect_and_describe(img2, method=method)
    
    matches = knn_ratio_match(d1, d2, k=2, ratio=PARAMS['ratio_test'],
                              matcher=PARAMS['matcher'], descriptor_type=method)
    
    print(f"Feature detection: {len(k1)} and {len(k2)} keypoints")
    print(f"Matches after ratio test: {len(matches)}\n")
    
    if len(matches) < 4:
        print("Not enough matches for testing")
        return None
    
    pts1, pts2 = matches_to_points(k1, k2, matches)
    
    # Test different parameter combinations
    test_configs = [
        # (max_iters, inlier_thresh, description)
        (1000, 3.0, "Baseline: 1000 iters, 3.0px thresh"),
        (2000, 3.0, "More iterations: 2000 iters, 3.0px thresh"),
        (3000, 3.0, "Even more iterations: 3000 iters, 3.0px thresh"),
        (2000, 2.0, "Tighter threshold: 2000 iters, 2.0px thresh"),
        (3000, 2.0, "Current settings: 3000 iters, 2.0px thresh"),
        (2000, 1.0, "Very strict: 2000 iters, 1.0px thresh"),
        (5000, 2.0, "Many iterations: 5000 iters, 2.0px thresh"),
    ]
    
    results = []
    
    for max_iters, inlier_thresh, desc in test_configs:
        H, inliers = ransac_homography(
            pts2, pts1,
            sample_size=4,
            max_iters=max_iters,
            inlier_thresh=inlier_thresh
        )
        
        if H is not None and inliers is not None:
            inlier_count = int(inliers.sum())
            inlier_ratio = inlier_count / len(matches) if len(matches) > 0 else 0
            
            # Compute median reprojection error for inliers
            in_idx = inliers.astype(bool)
            if in_idx.sum() >= 4:
                X1 = np.hstack([pts1[in_idx], np.ones((in_idx.sum(), 1))])
                X2 = np.hstack([pts2[in_idx], np.ones((in_idx.sum(), 1))])
                proj2 = (H @ X1.T).T
                proj2 = proj2[:, :2] / np.clip(proj2[:, 2:3], 1e-12, None)
                e12 = np.sqrt(np.sum((proj2 - pts2[in_idx])**2, axis=1))
                
                try:
                    Hinv = np.linalg.inv(H)
                    proj1 = (Hinv @ X2.T).T
                    proj1 = proj1[:, :2] / np.clip(proj1[:, 2:3], 1e-12, None)
                    e21 = np.sqrt(np.sum((proj1 - pts1[in_idx])**2, axis=1))
                    median_error = float(np.median((e12 + e21) / 2))
                except:
                    median_error = float(np.median(e12))
            else:
                median_error = float('inf')
        else:
            inlier_count = 0
            inlier_ratio = 0.0
            median_error = float('inf')
        
        results.append({
            'max_iters': max_iters,
            'inlier_thresh': inlier_thresh,
            'description': desc,
            'inliers': inlier_count,
            'total_matches': len(matches),
            'inlier_ratio': round(inlier_ratio * 100, 1),
            'median_error_px': round(median_error, 2) if median_error != float('inf') else 'N/A'
        })
    
    # Create DataFrame for display
    df = pd.DataFrame(results)
    print(f"\n{'='*80}")
    print(f"RANSAC Ablation Study: {scene_name} ({method})")
    print(f"{'='*80}\n")
    
    return df

ablation_df = ransac_ablation_study(scene_name='v_circus', method='SIFT')
display(ablation_df)

Testing RANSAC parameters on v_circus (images 1 and 2)
Feature detection: 4000 and 3675 keypoints
Matches after ratio test: 1631


RANSAC Ablation Study: v_circus (SIFT)



,max_iters,inlier_thresh,description,inliers,total_matches,inlier_ratio,median_error_px
0,1000,3.0,"Baseline: 1000 iters, 3.0px thresh",1557,1631,95.5,216.85
1,2000,3.0,"More iterations: 2000 iters, 3.0px thresh",1557,1631,95.5,216.85
2,3000,3.0,"Even more iterations: 3000 iters, 3.0px thresh",1557,1631,95.5,216.85
3,2000,2.0,"Tighter threshold: 2000 iters, 2.0px thresh",1540,1631,94.4,215.94
4,3000,2.0,"Current settings: 3000 iters, 2.0px thresh",1538,1631,94.3,216.03
5,2000,1.0,"Very strict: 2000 iters, 1.0px thresh",1363,1631,83.6,210.30
6,5000,2.0,"Many iterations: 5000 iters, 2.0px thresh",1536,1631,94.2,216.02


## 10. AR skeleton

In [ ]:

def central_crop_to_aspect(img, target_aspect):
    h, w = img.shape[:2]
    cur = w / h
    if abs(cur - target_aspect) < 1e-3:
        return img
    if cur > target_aspect:
        new_w = int(target_aspect * h)
        x0 = (w - new_w) // 2
        return img[:, x0:x0+new_w]
    else:
        new_h = int(w / target_aspect)
        y0 = (h - new_h) // 2
        return img[y0:y0+new_h, :]

def smooth_quad(prev_quad, new_quad, alpha=0.85):
    # EMA in image-space
    if prev_quad is None:
        return new_quad.copy()
    return (alpha * prev_quad + (1.0 - alpha) * new_quad).astype(np.float32)

def warp_src_to_quad(src_resized, quad_dst, frame_size):
    """
    Map the cover-sized source to the smoothed quad in the frame
    using our DLT-based estimate_homography.
    """
    h_c, w_c = src_resized.shape[:2]
    src_corners = np.array([[0,0],[w_c,0],[w_c,h_c],[0,h_c]], dtype=np.float64)
    dst_corners = quad_dst.astype(np.float64)

    # use our normalized DLT
    H_src_to_frame = estimate_homography(src_corners, dst_corners)
    if H_src_to_frame is None or not np.isfinite(H_src_to_frame).all():
        return None

    warped = cv2.warpPerspective(src_resized, H_src_to_frame.astype(np.float32), frame_size)
    return warped

def compute_frame_homography(cover_gray, frame_rgb,
                                          min_inliers=30, max_med_err=6.0):
    # detect/describe on cover and frame
    k1, d1 = detect_and_describe(cv2.cvtColor(cover_gray, cv2.COLOR_GRAY2RGB), method='SIFT')
    k2, d2 = detect_and_describe(frame_rgb, method='SIFT')
    if d1 is None or d2 is None or len(k1) < 4 or len(k2) < 4:
        return None, 0, np.inf

    matches = knn_ratio_match(d1, d2, k=2, ratio=PARAMS['ratio_test'],
                              matcher=PARAMS['matcher'], descriptor_type='SIFT')
    if len(matches) < 4:
        return None, 0, np.inf

    pts1, pts2 = matches_to_points(k1, k2, matches)
    H, inliers = ransac_homography(
        pts1, pts2,
        sample_size=PARAMS['ransac']['sample_size'],
        max_iters=PARAMS['ransac']['max_iters'],
        inlier_thresh=PARAMS['ransac']['inlier_thresh'],
    )
    if H is None or inliers is None:
        return None, 0, np.inf

    # median symmetric transfer error as a quality proxy
    in_idx = inliers.astype(bool)
    if in_idx.sum() < 4:
        return None, int(in_idx.sum()), np.inf

    # reuse symmetric error from ransac
    X1 = np.hstack([pts1[in_idx], np.ones((in_idx.sum(),1))])
    X2 = np.hstack([pts2[in_idx], np.ones((in_idx.sum(),1))])
    proj2 = (H @ X1.T).T
    proj2 = proj2[:, :2] / np.clip(proj2[:, 2:3], 1e-12, None)
    e12 = np.sum((proj2 - pts2[in_idx])**2, axis=1)
    try:
        Hinv = np.linalg.inv(H)
        proj1 = (Hinv @ X2.T).T
        proj1 = proj1[:, :2] / np.clip(proj1[:, 2:3], 1e-12, None)
        e21 = np.sum((proj1 - pts1[in_idx])**2, axis=1)
    except np.linalg.LinAlgError:
        e21 = np.full(e12.shape, np.inf)
    med_err = float(np.median(e12 + e21))

    # basic accept/reject
    if in_idx.sum() < min_inliers or med_err > (max_med_err**2):
        return None, int(in_idx.sum()), med_err

    return H, int(in_idx.sum()), med_err

def run_ar():
    book_path = os.path.join(DATA_AR_DIR, 'book.mov')
    src_path  = os.path.join(DATA_AR_DIR, 'ar_source.mov')
    cover_path= os.path.join(DATA_AR_DIR, 'cv_cover.jpg')

    if not (os.path.exists(book_path) and os.path.exists(src_path) and os.path.exists(cover_path)):
        raise FileNotFoundError('Missing AR data in ar_data/')

    cap_book = cv2.VideoCapture(book_path)
    cap_src  = cv2.VideoCapture(src_path)
    cover_rgb = imread_color(cover_path)
    cover_gray = cv2.cvtColor(cover_rgb, cv2.COLOR_RGB2GRAY)

    out_w = int(cap_book.get(cv2.CAP_PROP_FRAME_WIDTH))
    out_h = int(cap_book.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps   = cap_book.get(cv2.CAP_PROP_FPS) or 25.0
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(AR_OUT_VIDEO, fourcc, fps, (out_w, out_h))

    target_aspect = cover_rgb.shape[1] / cover_rgb.shape[0]
    frame_idx = 0

    last_good_quad = None
    lost_budget = 0
    LOST_BUDGET_MAX = 15

    # precompute fixed source in cover size
    def next_src_frame():
        ok, bgr = cap_src.read()
        if not ok:
            cap_src.set(cv2.CAP_PROP_POS_FRAMES, 0)
            ok, bgr = cap_src.read()
            if not ok:
                return None
        src_rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        src_rgb = central_crop_to_aspect(src_rgb, target_aspect)
        return cv2.resize(src_rgb, (cover_rgb.shape[1], cover_rgb.shape[0]))

    src_resized = next_src_frame()
    if src_resized is None:
        raise RuntimeError("Failed to read ar_source.mov")

    while True:
        ret_book, frame_bgr = cap_book.read()
        if not ret_book:
            break
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

        # Try to estimate H for this frame
        H, ninl, mederr = compute_frame_homography(
            cover_gray, frame_rgb, min_inliers=30, max_med_err=6.0
        )

        quad_this = None
        if H is not None:
            # cover -> frame corners
            h_c, w_c = cover_rgb.shape[:2]
            cover_corners = np.float32([[0,0],[w_c,0],[w_c,h_c],[0,h_c]])[None,:,:]
            quad_this = cv2.perspectiveTransform(cover_corners, H.astype(np.float32))[0]

            # area > small threshold and within reasonable bounds
            area = cv2.contourArea(quad_this.astype(np.float32))
            if area < 200:  # too tiny = almost off-screen
                quad_this = None

        if quad_this is None:
            # no good detection: reuse last pose for a bit to avoid flicker
            if last_good_quad is None or lost_budget >= LOST_BUDGET_MAX:
                # just write the original frame and reset
                writer.write(cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR))
                lost_budget = min(LOST_BUDGET_MAX, lost_budget + 1)
                frame_idx += 1
                if frame_idx % 30 == 0:
                    print('Processed', frame_idx, 'frames (tracking lost)')
                continue
            else:
                lost_budget += 1
                quad_smooth = last_good_quad.copy()
        else:
            # good detection: smooth with EMA and reset lost budget
            quad_smooth = smooth_quad(last_good_quad, quad_this, alpha=0.85)
            lost_budget = 0

        # Warp the source into smoothed quad
        warped = warp_src_to_quad(src_resized, quad_smooth, (out_w, out_h))
        if warped is None:
            writer.write(cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR))
            frame_idx += 1
            continue

        # Blend with a convex mask
        mask = np.zeros((out_h, out_w), dtype=np.uint8)
        cv2.fillConvexPoly(mask, quad_smooth.astype(np.int32), 255)
        mask3 = (mask > 0).astype(np.uint8)

        out_frame = simple_linear_blend(frame_rgb, warped, mask3)
        writer.write(cv2.cvtColor(out_frame, cv2.COLOR_RGB2BGR))

        last_good_quad = quad_smooth  # update after writing

        # step source
        tmp = next_src_frame()
        if tmp is not None:
            src_resized = tmp

        frame_idx += 1
        if frame_idx % 30 == 0:
            print(f'Processed {frame_idx} frames; inliers={ninl}, med_err≈{mederr:.2f}')

    cap_book.release()
    cap_src.release()
    writer.release()
    print('Saved AR video to:', AR_OUT_VIDEO)

run_ar()


Processed 30 frames; inliers=460, med_err≈0.23
Processed 60 frames; inliers=467, med_err≈0.22
Processed 90 frames; inliers=455, med_err≈0.24
Processed 120 frames; inliers=465, med_err≈0.24
Processed 150 frames; inliers=470, med_err≈0.23
Processed 180 frames; inliers=439, med_err≈0.21
Processed 210 frames; inliers=438, med_err≈0.27
Processed 240 frames; inliers=455, med_err≈0.33
Processed 270 frames; inliers=472, med_err≈0.36
Processed 300 frames; inliers=445, med_err≈0.46
Processed 330 frames; inliers=449, med_err≈0.44
Processed 360 frames; inliers=430, med_err≈0.40
Processed 390 frames; inliers=420, med_err≈0.29
Processed 420 frames; inliers=418, med_err≈0.23
Processed 450 frames; inliers=471, med_err≈0.21
Processed 480 frames; inliers=418, med_err≈0.25
Processed 510 frames; inliers=449, med_err≈0.21
Processed 540 frames; inliers=440, med_err≈0.25
Processed 570 frames; inliers=477, med_err≈0.25
Processed 600 frames; inliers=478, med_err≈0.26
Processed 630 frames; inliers=451, med_err≈